In [ ]:
from pathlib import Path
import sys, zipfile, numpy as np, pandas as pd

def find_data_dir():
    local_candidates = [Path.cwd() / "ai_model" / "data" / "single_sensor", Path.cwd().parent / "data" / "single_sensor"]
    for candidate in local_candidates:
        if candidate.is_dir() and list(candidate.glob("*.csv")):
            return candidate
    kaggle_root = Path("/kaggle/input")
    csv_files = list(kaggle_root.rglob("*.csv")) if kaggle_root.is_dir() else []
    if csv_files:
        return csv_files[0].parent
    zip_files = list(kaggle_root.rglob("*.zip")) if kaggle_root.is_dir() else []
    if zip_files:
        output = Path("/kaggle/working/single_sensor_dataset")
        output.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_files[0]) as archive:
            archive.extractall(output)
        extracted_csv = list(output.rglob("*.csv"))
        if extracted_csv:
            return extracted_csv[0].parent
    raise FileNotFoundError("Upload the single_sensor ZIP as a Kaggle Dataset.")

DATA_DIR = find_data_dir()
REPO_ROOT = next((candidate for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working")] if (candidate / "ai_model" / "feature_engineering.py").exists()), Path.cwd())
ROOT = REPO_ROOT
sys.path.insert(0, str(REPO_ROOT))

try:
    from ai_model.feature_engineering import load_recordings, FEATURE_COLUMNS
except ModuleNotFoundError:
    FEATURE_COLUMNS = ["distance_cm", "approach_speed_cm_s"]
    LABELS = {"SAFE": 0, "APPROACHING": 1, "DANGER": 2, "EMERGENCY": 3, "RETREATING": 0}

    def load_recordings(folder):
        frames = []
        for csv_path in sorted(Path(folder).glob("*.csv")):
            frame = pd.read_csv(csv_path)
            frame["source_file"] = csv_path.name
            frames.append(frame)
        if not frames:
            raise FileNotFoundError(f"No CSV files found in {folder}")
        frame = pd.concat(frames, ignore_index=True)
        for column in ["distance_cm", "elapsed_s"]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame = frame.dropna(subset=["distance_cm", "elapsed_s"]).copy()
        grouped = frame.groupby("source_file", sort=False)
        frame["approach_speed_cm_s"] = (grouped["distance_cm"].diff() / grouped["elapsed_s"].diff()).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        frame["label_id"] = frame["label"].astype(str).str.upper().map(LABELS).fillna(0).astype(int)
        return frame


# Single-sensor HRC Safety Log Ã¢â‚¬â€ Isolation Forest

HuÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n Isolation Forest trÃƒÂªn dataset HC-SR04 mÃ¡Â»â€ºi, chÃ¡Â»â€° cÃƒÂ³ `distance_cm` vÃƒÂ  `approach_speed_cm_s`. NgÃ†Â°Ã¡Â»Â¡ng an toÃƒÂ n cÃ¡Â»Â©ng vÃ¡ÂºÂ«n Ã„â€˜Ã†Â°Ã¡Â»Â£c Ã„â€˜ÃƒÂ¡nh giÃƒÂ¡ Ã„â€˜Ã¡Â»â„¢c lÃ¡ÂºÂ­p: `<=30 cm` lÃƒÂ  EMERGENCY vÃƒÂ  tÃ¡Â»Â± yÃƒÂªu cÃ¡ÂºÂ§u E-Stop; `31Ã¢â‚¬â€œ60 cm` lÃƒÂ  WARNING.

In [ ]:
from pathlib import Path
import sys, json, joblib
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# load_recordings and FEATURE_COLUMNS are initialized by the Kaggle/local setup cell

In [ ]:
data = load_recordings(DATA_DIR)
print('shape:', data.shape)
print('features:', FEATURE_COLUMNS)
print(data['label'].value_counts())
data[['distance_cm', 'approach_speed_cm_s', 'label']].head()

In [ ]:
groups = data['source_file']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(data, data['label_id'], groups=groups))
X_train, X_test = data.iloc[train_idx][FEATURE_COLUMNS], data.iloc[test_idx][FEATURE_COLUMNS]
y_train, y_test = data.iloc[train_idx]['label_id'], data.iloc[test_idx]['label_id']
normal = X_train[y_train == 0]
model = Pipeline([('scale', StandardScaler()), ('detector', IsolationForest(n_estimators=200, contamination=0.08, random_state=42))])
model.fit(normal)
predicted = (model.predict(X_test) == -1).astype(int)
actual = (y_test > 0).astype(int)
print(classification_report(actual, predicted, labels=[0, 1], target_names=['SAFE', 'ANOMALY'], zero_division=0))
print(confusion_matrix(actual, predicted, labels=[0, 1]))

In [ ]:
artifact = {
    'model': model,
    'feature_columns': FEATURE_COLUMNS,
    'thresholds_cm': {'warning': 60.0, 'danger': 30.0},
    'training_rows': int(len(normal)),
    'architecture': 'single_sensor_isolation_forest',
}
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else ROOT / 'ai_model' / 'artifacts'
output_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, output_dir / 'single_sensor_isolation_forest.joblib')
(output_dir / 'single_sensor_isolation_forest_metrics.json').write_text(json.dumps({'test_rows': len(X_test), 'normal_train_rows': len(normal)}, indent=2), encoding='utf-8')
print('saved:', output_dir / 'single_sensor_isolation_forest.joblib')